# IMPORTS

In [2]:
import pickle as pkl
from torch import nn
import os
import random
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch
import torch.nn.functional as F
import plotly.graph_objects as go
from matplotlib import cm
from datasets import Dataset, DatasetDict, Features, Value, Sequence
from huggingface_hub import HfApi, login
import pandas as pd
from tqdm import tqdm
import shutil
from datetime import datetime


# PARAMS

In [ ]:
# PARAMS
max_length = 50

parent_folder = "nanos_networkx_small"  # Update this to your data path - this is relative!
chunk_length = 50
max_proteins = 3000  # Limit number of proteins for faster execution
batch_size = 32
lr = 1e-4 # learning rate
num_epochs = 10
min_epochs = 1
patience = 30

# More aggressive subsequence parameters
min_subseq_length = 20  # Even smaller minimum subsequence length
step_size = 1  # Much smaller step size for more overlap
subgraph_limit = max_proteins * 300 # max number of subgraphs


model_path = "dual_output_gran_model.pt"  # Path for saving/loading model





# Load data -
some of this is just sanity checks and global things needed later, adjacency matrices, AA sequences

In [7]:

# Load protein graph data directly
try:
    full_graphs, full_sequences, subgraphs, subsequences = load_protein_graph_data(
        parent_folder, max_proteins, chunk_length
    )
except Exception as e:
    print(f"Error loading graph data: {e}")
    print("Current directory contains:", os.listdir())



# First graph debug
print("First few nodes of first graph:")
first_graph = full_graphs[0]
for i, node in enumerate(sorted(first_graph.nodes())[:5]):
    print(f"Node {node} attributes: {first_graph.nodes[node]}")

# Define a standard set of amino acids (all 20 standard ones)
STANDARD_AA = ['ALA', 'ARG', 'ASN', 'ASP', 'CYS', 'GLN', 'GLU', 'GLY', 'HIS', 'ILE',
               'LEU', 'LYS', 'MET', 'PHE', 'PRO', 'SER', 'THR', 'TRP', 'TYR', 'VAL', 'X']

# Get unique amino acids from sequences but ensure we have at least the standard 20
UNIQUE_AA = set()
for seq in full_sequences:
    UNIQUE_AA.update(seq)

# Merge with standard AAs
UNIQUE_AA = sorted(list(set(UNIQUE_AA).union(set(STANDARD_AA))))
print(f"Unique amino acids: {len(UNIQUE_AA)}")
print(f"Amino acids found: {', '.join(UNIQUE_AA)}")

# Prepare data for training using graph data
aa_sequences, adjacency_matrices, node_features = prepare_graph_data_for_training(
    subgraphs, subsequences, UNIQUE_AA
)

print(f"Prepared data: {len(aa_sequences)} sequences, {len(adjacency_matrices)} adjacency matrices")


Found 3015 protein folders
Limited to 3000 folders due to max_proteins setting
Error loading nanos_networkx_small/5JDS_nanobody_B/5JDS_nanobody_B_graph.pkl: Ran out of input
Error loading nanos_networkx_small/8RW9_nanobody_C/8RW9_nanobody_C_graph.pkl: Ran out of input
Error loading nanos_networkx_small/8YBO_nanobody_B/8YBO_nanobody_B_graph.pkl: Ran out of input
Error loading nanos_networkx_small/6IBL_nanobody_C/6IBL_nanobody_C_graph.pkl: Ran out of input
Error loading nanos_networkx_small/8FQ7_nanobody_A/8FQ7_nanobody_A_graph.pkl: Ran out of input
Error loading nanos_networkx_small/2P43_nanobody_B/2P43_nanobody_B_graph.pkl: Ran out of input
Error loading nanos_networkx_small/7LVW_nanobody_I/7LVW_nanobody_I_graph.pkl: Ran out of input
Error loading nanos_networkx_small/7DSS_nanobody_A/7DSS_nanobody_A_graph.pkl: Ran out of input
Successfully loaded 2965 protein graphs, 8 failed

Subsequence Generation Statistics:
Total possible subsequences: 358689
Actually generated subsequences: 358689

In [12]:


def extract_gran_data(full_graphs, full_sequences, unique_aa):
    """Extract only essential data for GRAN - full proteins without subsequences"""

    # Create full proteins data
    full_proteins_data = []
    for i, (graph, seq) in enumerate(zip(full_graphs, full_sequences)):
        # Extract graph data
        adj_matrix = None
        if hasattr(graph, 'edges'):
            # Get adjacency matrix from graph
            import networkx as nx
            adj_matrix = nx.to_numpy_array(graph).tolist()

        # Extract meiler features if available
        node_features = []
        if hasattr(graph, 'nodes'):
            sorted_nodes = sorted(graph.nodes(), key=lambda x: int(graph.nodes[x].get('residue_number', 0)))
            for node in sorted_nodes:
                if 'meiler' in graph.nodes[node]:
                    # Convert pandas Series to list
                    meiler_data = graph.nodes[node]['meiler']
                    if hasattr(meiler_data, 'tolist'):
                        node_features.append(meiler_data.tolist())
                    else:
                        # In case it's already a list or array
                        node_features.append(list(meiler_data))
                else:
                    # Fallback to one-hot encoding if meiler features not available
                    aa = graph.nodes[node].get('residue_name', 'X')
                    aa_idx = unique_aa.index(aa) if aa in unique_aa else unique_aa.index('X') if 'X' in unique_aa else 0
                    one_hot = [0] * len(unique_aa)
                    one_hot[aa_idx] = 1
                    node_features.append(one_hot)

        protein_dict = {
            'sequence': seq,
            'sequence_length': len(seq),
            'graph_nodes': list(graph.nodes()) if hasattr(graph, 'nodes') else None,
            'graph_edges': list(graph.edges()) if hasattr(graph, 'edges') else None,
            'adjacency_matrix': adj_matrix,
            'node_features': node_features,
            'protein_id': f"protein_{i}"
        }
        full_proteins_data.append(protein_dict)

    # Create metadata
    metadata = {
        'unique_amino_acids': unique_aa,
        'num_proteins': len(full_proteins_data),
        'creation_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        'average_protein_length': sum(len(seq) for seq in full_sequences) / len(full_sequences) if full_sequences else 0
    }

    return full_proteins_data, metadata

def upload_gran_dataset_to_hf(full_graphs, full_sequences, unique_aa,
                              dataset_name="gran-protein-structures",
                              username="alexchilton", save_local=True):
    """Upload essential GRAN protein dataset to Hugging Face account"""
    # Login to Hugging Face
    login()

    # Extract data
    print("Extracting GRAN data...")
    full_proteins_data, metadata = extract_gran_data(
        full_graphs, full_sequences, unique_aa
    )

    # Create dataset with explicit features
    print("Creating dataset...")

    # Define the features schema explicitly
    features = Features({
        'sequence': Sequence(Value('string')),
        'sequence_length': Value('int32'),
        'graph_nodes': Sequence(Value('string')),
        'graph_edges': Sequence(Sequence(Value('string'))),
        'adjacency_matrix': Sequence(Sequence(Value('float32'))),
        'node_features': Sequence(Sequence(Value('float32'))),
        'protein_id': Value('string')
    })

    protein_dataset = Dataset.from_pandas(pd.DataFrame(full_proteins_data), features=features)

    # Create a DatasetDict (single split for now)
    dataset_dict = DatasetDict({
        'train': protein_dataset
    })

    # Save locally before uploading if requested
    if save_local:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        local_dir = f"gran_protein_dataset_{timestamp}"
        print(f"Saving dataset locally to {local_dir}...")

        # Save using multiple formats for flexibility
        dataset_dict.save_to_disk(local_dir)

        # Also save raw data as pickle for backup
        raw_data = {
            'full_graphs': full_graphs,
            'full_sequences': full_sequences,
            'unique_aa': unique_aa,
            'metadata': metadata
        }
        with open(f"{local_dir}_raw_data.pkl", 'wb') as f:
            pkl.dump(raw_data, f)

        print(f"Local save complete. Files saved in {local_dir}")

    # Define a README.md content for the dataset card
    readme_content = f"""---
license: mit
task_categories:
- text-generation
- graph-ml
tags:
- protein
- graph-neural-network
- adjacency-matrix
- protein-structure
- nanobody
---

# GRAN Protein Structure Dataset

## Dataset Description

This dataset contains protein graph data for training Graph Recurrent Attention Networks (GRAN) for protein sequence and structure generation.

### Dataset Summary

- **Number of proteins:** {len(full_proteins_data)}
- **Average protein length:** {metadata['average_protein_length']:.1f} residues
- **Unique amino acids:** {len(metadata['unique_amino_acids'])}
- **Source:** Nanobody protein structures
- **Created by:** {username}
- **Date:** {metadata['creation_date']}

### Dataset Structure

Each protein entry contains:
- `sequence`: Complete amino acid sequence
- `sequence_length`: Total length of the protein
- `graph_nodes`: List of graph nodes (residue indices)
- `graph_edges`: List of graph edges (connections between residues)
- `adjacency_matrix`: Binary adjacency matrix representing contacts
- `node_features`: Features for each node (Meiler features or one-hot encoded residues)
- `protein_id`: Unique identifier

### Amino Acids

Available amino acids: {', '.join(metadata['unique_amino_acids'])}

### Usage

```python
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("{username}/{dataset_name}")

# Access a protein
protein = dataset['train'][0]
print(f"Sequence length: {{protein['sequence_length']}}")
print(f"Number of graph nodes: {{len(protein['graph_nodes'])}}")
print(f"Adjacency matrix shape: {{np.array(protein['adjacency_matrix']).shape}}")
```

### Training GRAN Model

This dataset is designed for training GRAN models that:
1. Generate both protein sequences and contact adjacency matrices
2. Model proteins as graphs with nodes (residues) and edges (contacts)
3. Use node features (Meiler descriptors or one-hot encoding)

### Citation

If you use this dataset, please cite:
```
@dataset{{gran_protein_structures,
  title={{GRAN Protein Structure Dataset}},
  author={{Alex Chilton}},
  year={{2025}},
  url={{https://huggingface.co/datasets/{username}/{dataset_name}}}
}}
```
"""

    # Upload to Hugging Face
    print(f"Uploading to {username}/{dataset_name}...")
    dataset_dict.push_to_hub(
        f"{username}/{dataset_name}",
        private=False,
        commit_message="Initial upload of GRAN protein structure dataset"
    )

    # Create and upload the README.md
    api = HfApi()
    api.upload_file(
        path_or_fileobj=readme_content.encode(),
        path_in_repo="README.md",
        repo_id=f"{username}/{dataset_name}",
        repo_type="dataset",
        commit_message="Add dataset card"
    )

    print(f"Successfully uploaded to https://huggingface.co/datasets/{username}/{dataset_name}")

    return dataset_dict



In [13]:
dataset = upload_gran_dataset_to_hf(
    full_graphs=full_graphs,
    full_sequences=full_sequences,
    unique_aa=UNIQUE_AA,
    dataset_name="gran-nanobody-proteins",
    username="alexchilton",
    save_local=True
)

Extracting GRAN data...
Creating dataset...
Saving dataset locally to gran_protein_dataset_20250504_160139...


Saving the dataset (0/1 shards):   0%|          | 0/2965 [00:00<?, ? examples/s]

Local save complete. Files saved in gran_protein_dataset_20250504_160139
Uploading to alexchilton/gran-nanobody-proteins...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Successfully uploaded to https://huggingface.co/datasets/alexchilton/gran-nanobody-proteins
